In [1]:
# CELDA 1 — Setup del notebook de Feature Engineering
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import unicodedata
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Paths
PROJECT_PATH = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
FIGURES_PATH = f'{PROJECT_PATH}/reports/figures'

# Cargar dataset del notebook anterior
df = pd.read_parquet(f'{PROCESSED_PATH}/dataset_eda.parquet')
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Operaciones únicas: {df['Nro. Ope.'].nunique():,}")
print(f"Rango temporal: {df['Fecha_Imputada'].min().strftime('%Y-%m-%d')} → {df['Fecha_Imputada'].max().strftime('%Y-%m-%d')}")

Mounted at /content/drive
Dataset cargado: 14,705 filas × 56 columnas
Operaciones únicas: 1,252
Rango temporal: 2020-07-01 → 2026-07-01


In [2]:
# CELDA 2 — Excluir notas de crédito (montos negativos y ceros)
n_inicial = len(df)

# Diagnóstico antes del descarte
neg = df[df['Importe_Total_PEN'] < 0]
ceros = df[df['Importe_Total_PEN'] == 0]
nans = df[df['Importe_Total_PEN'].isna()]

print(f"📊 Diagnóstico antes del descarte:")
print(f"  Total filas:              {n_inicial:,}")
print(f"  Filas negativas (NC):     {len(neg):,}  (Monto: S/ {neg['Importe_Total_PEN'].sum():,.0f})")
print(f"  Filas con monto cero:     {len(ceros):,}")
print(f"  Filas con monto NaN:      {len(nans):,}")

# Aplicar filtro: solo facturas con monto positivo
df = df[df['Importe_Total_PEN'] > 0].copy().reset_index(drop=True)

print(f"\n✓ Dataset filtrado: {len(df):,} filas (se excluyeron {n_inicial - len(df):,})")
print(f"  Porcentaje conservado: {len(df)/n_inicial*100:.2f}%")

📊 Diagnóstico antes del descarte:
  Total filas:              14,705
  Filas negativas (NC):     32  (Monto: S/ -115,741)
  Filas con monto cero:     13
  Filas con monto NaN:      0

✓ Dataset filtrado: 14,660 filas (se excluyeron 45)
  Porcentaje conservado: 99.69%


In [3]:
# CELDA 3 — Features temporales a partir de Fecha_Imputada
df['año']               = df['Fecha_Imputada'].dt.year
df['mes']               = df['Fecha_Imputada'].dt.month
df['trimestre']         = df['Fecha_Imputada'].dt.quarter
df['semana_año']        = df['Fecha_Imputada'].dt.isocalendar().week.astype(int)
df['dia_semana']        = df['Fecha_Imputada'].dt.dayofweek  # 0=lunes, 6=domingo

# Tiempo lineal desde el inicio del histórico (útil para capturar tendencias)
fecha_inicio = df['Fecha_Imputada'].min()
df['dias_desde_inicio'] = (df['Fecha_Imputada'] - fecha_inicio).dt.days

# Flag de temporada alta de berries en Perú (agosto - diciembre)
df['es_temporada_alta'] = df['mes'].isin([8, 9, 10, 11, 12]).astype(int)

print("Features temporales creadas:")
nuevas = ['año', 'mes', 'trimestre', 'semana_año', 'dia_semana', 'dias_desde_inicio', 'es_temporada_alta']
print(df[nuevas].describe().round(1).to_string())

print(f"\nDistribución por mes (cuántas facturas por mes calendario):")
print(df['mes'].value_counts().sort_index().to_string())

print(f"\nProporción en temporada alta vs baja:")
print(df['es_temporada_alta'].value_counts(normalize=True).round(3).to_string())

Features temporales creadas:
           año      mes  trimestre  semana_año  dia_semana  dias_desde_inicio  es_temporada_alta
count  14660.0  14660.0    14660.0     14660.0     14660.0            14660.0            14660.0
mean    2022.8      6.7        2.7        26.8         2.3             1027.5                0.3
std        1.6      3.2        1.1        13.7         1.5              599.4                0.5
min     2020.0      1.0        1.0         1.0         0.0                0.0                0.0
25%     2022.0      4.0        2.0        16.0         1.0              572.0                0.0
50%     2022.0      7.0        3.0        26.0         2.0              852.0                0.0
75%     2024.0      9.0        3.0        38.0         4.0             1617.0                1.0
max     2026.0     12.0        4.0        52.0         6.0             2191.0                1.0

Distribución por mes (cuántas facturas por mes calendario):
mes
1     1009
2     1209
3     1104


In [4]:
# CELDA 4 — Consolidar duplicados ortográficos en categóricas
def normalizar_categorica(s):
    """Mayúsculas, sin tildes, sin puntos, espacios colapsados."""
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    # Sacar tildes (NFD descompone tilde+letra, después filtramos las tildes)
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    # Sacar puntos y comas
    s = s.replace('.', '').replace(',', '')
    # Colapsar espacios múltiples
    s = ' '.join(s.split())
    return s

# Columnas a normalizar (ajustá los nombres si en tu df aparecen distintos)
columnas_a_consolidar = ['Proveedor Principal', 'Producto', 'AGENCIA DE ADUANA', 'ACREEDOR', 'Proveedor']

print(f"{'Columna':<25} {'Antes':>10} {'Después':>10} {'Reducción':>12}")
print("-" * 60)

for col in columnas_a_consolidar:
    if col not in df.columns:
        print(f"{col:<25}   (no existe en el dataset)")
        continue
    antes = df[col].nunique()
    df[f'{col}_norm'] = df[col].apply(normalizar_categorica)
    despues = df[f'{col}_norm'].nunique()
    reduccion = antes - despues
    pct = reduccion / antes * 100 if antes else 0
    print(f"{col:<25} {antes:>10,} {despues:>10,} {reduccion:>9,} ({pct:.1f}%)")

print(f"\nDataset ahora tiene {df.shape[1]} columnas")

Columna                        Antes    Después    Reducción
------------------------------------------------------------
Proveedor Principal              164        155         9 (5.5%)
Producto                         379        364        15 (4.0%)
AGENCIA DE ADUANA                 23         21         2 (8.7%)
ACREEDOR                          77         66        11 (14.3%)
Proveedor                        175        148        27 (15.4%)

Dataset ahora tiene 68 columnas


In [5]:
# CELDA 5 — Agregar flag fecha_original (1 = venía del archivo, 0 = fue imputada al 1-jul)
df['fecha_original'] = df['Fecha de Emisión Doc'].notna().astype(int)

print(f"Distribución del flag fecha_original:")
print(df['fecha_original'].value_counts().to_string())
print(f"\n  Filas con fecha REAL del archivo:  {(df['fecha_original']==1).sum():,} ({(df['fecha_original']==1).mean()*100:.1f}%)")
print(f"  Filas con fecha imputada (1-jul):  {(df['fecha_original']==0).sum():,} ({(df['fecha_original']==0).mean()*100:.1f}%)")

# Verificación: mirar la distribución de mes SOLO para fechas reales
print(f"\n📅 Distribución por mes — SOLO fechas reales (sin el artefacto):")
print(df[df['fecha_original']==1]['mes'].value_counts().sort_index().to_string())

Distribución del flag fecha_original:
fecha_original
1    10009
0     4651

  Filas con fecha REAL del archivo:  10,009 (68.3%)
  Filas con fecha imputada (1-jul):  4,651 (31.7%)

📅 Distribución por mes — SOLO fechas reales (sin el artefacto):
mes
1     1009
2     1209
3     1104
4      460
5      333
6      455
7      692
8      639
9      928
10    1040
11    1114
12    1026


In [ ]:
# CAMBIO A — Asignar pesos de muestra para down-weightear filas con fecha imputada
# Las 4,651 filas imputadas al 1-jul tienen fecha incorrecta → todas las features
# estacionales también son incorrectas. Reducirlas a 30% del peso de una fila real
# hace que el modelo aprenda estacionalidad casi exclusivamente de los 10,009
# registros con fecha real, sin descartar la información de precio que sí contienen.
IMPUTED_WEIGHT = 0.3  # hiperparámetro: filas imputadas valen 30% de una fila real

df['sample_weight'] = np.where(df['fecha_original'] == 1, 1.0, IMPUTED_WEIGHT)

print(f"✓ sample_weight asignado:")
vc = df['sample_weight'].value_counts()
print(f"  Peso 1.0  (fecha real):        {vc.get(1.0, 0):,} filas")
print(f"  Peso {IMPUTED_WEIGHT} (fecha imputada):  {vc.get(IMPUTED_WEIGHT, 0):,} filas")

In [ ]:
# CELDA 6 — Features derivadas del despacho
# Flag de peso disponible (importante porque solo está en 2025-26)
df['peso_disponible'] = df['Peso Bruto (kg)'].notna().astype(int)

# Densidad de bultos por contenedor
df['bultos_por_contenedor'] = np.where(
    (df['Cantidad de Contenedores'] > 0) & df['Cantidad de Bultos (BULKS)'].notna(),
    df['Cantidad de Bultos (BULKS)'] / df['Cantidad de Contenedores'],
    np.nan
)

# Peso por contenedor (solo cuando ambos existen)
df['peso_por_contenedor'] = np.where(
    (df['Cantidad de Contenedores'] > 0) & df['Peso Bruto (kg)'].notna(),
    df['Peso Bruto (kg)'] / df['Cantidad de Contenedores'],
    np.nan
)

# CAMBIO C — Recortar outliers usando solo filas con fecha real (evita que el
# artefacto de julio infle el percentil de referencia).
p99_bultos = df.loc[df['fecha_original'] == 1, 'bultos_por_contenedor'].quantile(0.99)
p99_peso   = df.loc[df['fecha_original'] == 1, 'peso_por_contenedor'].dropna().quantile(0.99)
df['bultos_por_contenedor'] = df['bultos_por_contenedor'].clip(upper=p99_bultos)
df['peso_por_contenedor']   = df['peso_por_contenedor'].clip(upper=p99_peso)
print(f"Outliers recortados: bultos_por_contenedor ≤ {p99_bultos:.1f}, "
      f"peso_por_contenedor ≤ {p99_peso:.1f}")

# Flag binario de proyecto/programa
df['tiene_proyecto'] = (
    df['Proyecto'].notna() & (df['Proyecto'].astype(str).str.strip() != '-')
).astype(int)

# Agrupar Incoterm en familias (E, F, C, D)
def incoterm_familia(s):
    if pd.isna(s):
        return 'DESCONOCIDO'
    s = str(s).upper().strip()
    if s.startswith('E'):   return 'GRUPO_E'   # EXW
    if s.startswith('F'):   return 'GRUPO_F'   # FOB, FCA, FAS
    if s.startswith('C'):   return 'GRUPO_C'   # CFR, CIF, CPT, CIP
    if s.startswith('D'):   return 'GRUPO_D'   # DAP, DAT, DDP
    return 'OTRO'

df['incoterm_familia'] = df['Incoterm'].apply(incoterm_familia)

# Resumen
print("\nFeatures derivadas creadas:\n")

print("• peso_disponible (1 = tiene peso bruto):")
print(df['peso_disponible'].value_counts(normalize=True).round(3).to_string())

print("\n• bultos_por_contenedor (post-cap):")
print(df['bultos_por_contenedor'].describe().round(1).to_string())

print("\n• tiene_proyecto:")
print(df['tiene_proyecto'].value_counts(normalize=True).round(3).to_string())

print("\n• incoterm_familia:")
print(df['incoterm_familia'].value_counts().to_string())

print(f"\nDataset ahora tiene {df.shape[1]} columnas")

In [ ]:
# CELDA 7 — Target encoding histórico con decaimiento exponencial (EWM)
# CAMBIO D: Reemplaza la mediana expandida (igual peso 2020-2025) con una media
# exponencialmente ponderada donde datos de hace 1 año pesan ~50% vs datos recientes.
# Esto corrige la subestimación causada por precios más bajos en años anteriores.

# Asegurar que las columnas categóricas no tengan NaN
df['Proveedor_norm'] = df['Proveedor_norm'].fillna('DESCONOCIDO')
df['ACREEDOR_norm']  = df['ACREEDOR_norm'].fillna('DESCONOCIDO')

# Ordenar por fecha (crítico para que el EWM funcione cronológicamente)
df = df.sort_values('Fecha_Imputada').reset_index(drop=True)

HALF_LIFE_DAYS = 365  # datos de hace 1 año pesan ~50% vs datos recientes

def tarifa_ewm_por_grupo(subdf):
    """EWM en escala log para reducir influencia de valores extremos. shift(1) evita leakage."""
    subdf = subdf.sort_values('Fecha_Imputada').copy()
    log_vals = np.log1p(subdf['Importe_Total_PEN'])
    ewm_log = log_vals.ewm(halflife=f'{HALF_LIFE_DAYS}D',
                           times=subdf['Fecha_Imputada']).mean().shift(1)
    return np.expm1(ewm_log)

# Nivel 1 — EWM por (Proveedor × Concepto Canónico)
df['tarifa_hist_prov_concepto'] = (
    df.groupby(['Proveedor_norm', 'Concepto Canónico'], group_keys=False)
      .apply(tarifa_ewm_por_grupo)
)

# Nivel 2 — EWM por Concepto Canónico (fallback para proveedores nuevos)
df['tarifa_hist_concepto'] = (
    df.groupby('Concepto Canónico', group_keys=False)
      .apply(tarifa_ewm_por_grupo)
)

# Nivel 3 — EWM global (fallback para conceptos nuevos)
df_sorted_global = df.sort_values('Fecha_Imputada').copy()
log_global = np.log1p(df_sorted_global['Importe_Total_PEN'])
ewm_global = log_global.ewm(halflife=f'{HALF_LIFE_DAYS}D',
                             times=df_sorted_global['Fecha_Imputada']).mean().shift(1)
df['tarifa_hist_global'] = np.expm1(ewm_global).values

# Tarifa histórica final con cascada de fallbacks
df['tarifa_historica'] = (
    df['tarifa_hist_prov_concepto']
      .fillna(df['tarifa_hist_concepto'])
      .fillna(df['tarifa_hist_global'])
)

# Frequency encoding: cuántas veces vimos a este proveedor antes (también con shift)
df['proveedor_frecuencia'] = (
    df.groupby('Proveedor_norm').cumcount()
)

# Diagnóstico
print(f"📊 Cobertura de la tarifa histórica (EWM):")
print(f"  Nivel 1 (proveedor × concepto): {df['tarifa_hist_prov_concepto'].notna().sum():,} ({df['tarifa_hist_prov_concepto'].notna().mean()*100:.1f}%)")
print(f"  Nivel 2 (concepto):             {df['tarifa_hist_concepto'].notna().sum():,} ({df['tarifa_hist_concepto'].notna().mean()*100:.1f}%)")
print(f"  Final (cualquier nivel):        {df['tarifa_historica'].notna().sum():,} ({df['tarifa_historica'].notna().mean()*100:.1f}%)")

print(f"\n💰 Distribución de tarifa_historica (EWM):")
print(df['tarifa_historica'].describe().round(0).to_string())

correlacion = df[['tarifa_historica', 'Importe_Total_PEN']].dropna().corr().iloc[0, 1]
print(f"\n🔍 Correlación tarifa_historica (EWM) vs Importe_Total_PEN: {correlacion:.3f}")
print("   (objetivo: >0.45; si supera la mediana expandida anterior de 0.355, el cambio ayuda)")

In [ ]:
# CAMBIO E — Feature: ratio histórico del concepto dentro del despacho
# Captura qué fracción del costo total del despacho representa cada concepto históricamente.
# Ej: FLETE_INTERNACIONAL suele ser ~25% del total → ancla la magnitud relativa.

# Fracción real del concepto en su operación (para calcular el histórico)
total_por_op = df.groupby('Nro. Ope.')['Importe_Total_PEN'].transform('sum')
df['fraccion_concepto_op'] = df['Importe_Total_PEN'] / total_por_op.clip(lower=1)

# Mediana histórica expandida de esa fracción por concepto (shift=1 → no leakage)
df['ratio_hist_concepto'] = (
    df.groupby('Concepto Canónico')['fraccion_concepto_op']
      .transform(lambda x: x.expanding().median().shift(1))
)
# Fallback: mediana global del concepto para las primeras apariciones
df['ratio_hist_concepto'] = df['ratio_hist_concepto'].fillna(
    df.groupby('Concepto Canónico')['fraccion_concepto_op'].transform('median')
)

print("ratio_hist_concepto — fracción típica del costo total por concepto:")
print(df.groupby('Concepto Canónico')['ratio_hist_concepto'].median().round(3).sort_values(ascending=False).to_string())
print(f"\nCobertura: {df['ratio_hist_concepto'].notna().mean()*100:.1f}%")

# Nota: fraccion_concepto_op NO va en FEATURES (requiere conocer el total real → circular)
# Solo ratio_hist_concepto (histórico) va al modelo.

In [ ]:
# CELDA 8a-d — Estacionalidad sinusoidal, tránsito estimado, ruta de origen + CAMBIO B

# ── 8a: Codificación cíclica de mes y semana ──────────────────────────────────────
# sin/cos captura que mes 12 y mes 1 son continuos (no hay salto brusco numérico)
df['mes_sin']    = np.sin(2 * np.pi * df['mes']        / 12)
df['mes_cos']    = np.cos(2 * np.pi * df['mes']        / 12)
df['semana_sin'] = np.sin(2 * np.pi * df['semana_año'] / 52)
df['semana_cos'] = np.cos(2 * np.pi * df['semana_año'] / 52)

# Semanas hasta cierre de año fiscal (Perú: año fiscal = año calendario)
df['semanas_hasta_cierre'] = ((52 - df['semana_año']) % 52).astype(float)

# Flag para meses de cierre fiscal (nov-dic = apuro de gastos + cierre contable)
df['es_cierre_fiscal'] = df['mes'].isin([11, 12]).astype(int)

# ── 8b: Tiempo de tránsito estimado por modalidad ─────────────────────────────────
TRANSITO_POR_MODALIDAD = {
    'AIR': 3, 'AEREO': 3,
    'FCL': 25, 'LCL': 30,
    'RO-RO': 20, 'RORO': 20,
    'BREAK BULK': 35, 'BREAKBULK': 35,
}
def estimar_transito(modalidad):
    if pd.isna(modalidad):
        return 25
    m = str(modalidad).upper()
    for key, val in TRANSITO_POR_MODALIDAD.items():
        if key in m:
            return val
    return 25

col_modalidad = 'Modalidad (MODE Y TYPE)'
if col_modalidad in df.columns:
    df['dias_transito_estimado'] = df[col_modalidad].apply(estimar_transito).astype(float)
else:
    df['dias_transito_estimado'] = 25.0

# ── 8d: Grupo de ruta de origen (POL → familia geográfica) ───────────────────────
POL_GRUPOS = {
    'EUROPA':  ['VALENCIA', 'BARCELONA', 'ROTTERDAM', 'AMSTERDAM', 'BREMERHAVEN',
                'LE HAVRE', 'HAMBURG', 'ANTWERP', 'GENOVA', 'LIVORNO'],
    'ASIA':    ['COLOMBO', 'SHANGHAI', 'SHENZHEN', 'HONG KONG', 'SINGAPORE',
                'KAOHSIUNG', 'BUSAN', 'TOKYO', 'MUMBAI', 'CHENNAI'],
    'LATAM':   ['SANTIAGO', 'SAN ANTONIO', 'MIAMI', 'BOGOTA', 'QUITO',
                'GUAYAQUIL', 'SAO PAULO', 'BUENOS AIRES'],
    'NORTEAM': ['LOS ANGELES', 'NEW YORK', 'HOUSTON', 'SEATTLE'],
}
def clasificar_pol(pol):
    if pd.isna(pol):
        return 'DESCONOCIDO'
    pol_up = str(pol).upper()
    for grupo, puertos in POL_GRUPOS.items():
        if any(p in pol_up for p in puertos):
            return grupo
    return 'OTRO'

col_pol = next((c for c in df.columns if c.upper() == 'POL'), None)
if col_pol:
    df['ruta_origen_grupo'] = df[col_pol].apply(clasificar_pol)
else:
    df['ruta_origen_grupo'] = 'DESCONOCIDO'

# ── CAMBIO B: Neutralizar features estacionales en filas con fecha imputada ───────
# Para las 4,651 filas imputadas al 1-jul, mes_sin/mes_cos reportan "julio" aunque
# la factura puede ser de cualquier mes. Valores neutros son estrictamente mejores
# que valores incorrectos. Combinado con sample_weight=0.3, el modelo aprende
# estacionalidad casi exclusivamente de los 10,009 registros con fecha real.
IMPUTED_MASK = df['fecha_original'] == 0
df.loc[IMPUTED_MASK, 'mes_sin']              = 0.0
df.loc[IMPUTED_MASK, 'mes_cos']              = 0.0
df.loc[IMPUTED_MASK, 'semana_sin']           = 0.0
df.loc[IMPUTED_MASK, 'semana_cos']           = 0.0
df.loc[IMPUTED_MASK, 'semanas_hasta_cierre'] = 26.0   # punto medio del año
df.loc[IMPUTED_MASK, 'es_temporada_alta']    = 0       # desconocido → fuera de temporada
df.loc[IMPUTED_MASK, 'es_cierre_fiscal']     = 0       # desconocido → fuera de cierre

real_mask = ~IMPUTED_MASK
print(f"✓ Features sinusoidales creadas. En filas con fecha REAL:")
print(f"  mes_sin    ∈ [{df.loc[real_mask,'mes_sin'].min():.3f}, {df.loc[real_mask,'mes_sin'].max():.3f}]")
print(f"  semana_sin ∈ [{df.loc[real_mask,'semana_sin'].min():.3f}, {df.loc[real_mask,'semana_sin'].max():.3f}]")
print(f"\n✓ CAMBIO B: {IMPUTED_MASK.sum():,} filas imputadas → features estacionales = 0 (neutro)")
print(f"✓ ruta_origen_grupo: {df['ruta_origen_grupo'].value_counts().to_dict()}")
print(f"✓ dias_transito_estimado: mediana={df['dias_transito_estimado'].median():.0f}d")

## Mejoras P3-9: Estacionalidad Sinusoidal, Tránsito y Ruta

In [ ]:
# CELDA 9 — Guardar dataset modelable final
# Re-split desde df con TODAS las features ya computadas (corrige el bug original
# donde el split ocurría antes de las features sinusoidales y sample_weight).
cutoff_date = pd.Timestamp('2025-01-01')

# Garantizar sample_weight si CAMBIO A no corrió por algún motivo
if 'sample_weight' not in df.columns:
    IMPUTED_WEIGHT = 0.3
    df['sample_weight'] = np.where(df['fecha_original'] == 1, 1.0, IMPUTED_WEIGHT)
    print("⚠️ sample_weight calculado como fallback — verificar que CAMBIO A esté en el notebook")

train = df[df['Fecha_Imputada'] < cutoff_date].copy().reset_index(drop=True)
test  = df[df['Fecha_Imputada'] >= cutoff_date].copy().reset_index(drop=True)

train.to_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet', index=False)
test.to_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet', index=False)

print(f"✓ Guardado dataset_modelable_train.parquet")
print(f"  Tamaño:   {os.path.getsize(f'{PROCESSED_PATH}/dataset_modelable_train.parquet') / 1024:.1f} KB")
print(f"  Filas:    {len(train):,}  |  Columnas: {train.shape[1]}")
print(f"  sample_weight incluido:       {'sample_weight' in train.columns}")
print(f"  ratio_hist_concepto incluido: {'ratio_hist_concepto' in train.columns}")
print(f"  mes_sin incluido:             {'mes_sin' in train.columns}")
print(f"  ruta_origen_grupo incluido:   {'ruta_origen_grupo' in train.columns}")

print(f"\n✓ Guardado dataset_modelable_test.parquet")
print(f"  Tamaño:   {os.path.getsize(f'{PROCESSED_PATH}/dataset_modelable_test.parquet') / 1024:.1f} KB")
print(f"  Filas:    {len(test):,}  |  Columnas: {test.shape[1]}")

print(f"\n📊 Distribución de sample_weight en train:")
print(train['sample_weight'].value_counts().to_string())

# Verificación de todas las features esperadas
FEATURES_ESPERADAS = [
    'año', 'mes', 'trimestre', 'semana_año', 'dia_semana', 'dias_desde_inicio',
    'es_temporada_alta', 'fecha_original', 'sample_weight',
    'peso_disponible', 'bultos_por_contenedor', 'peso_por_contenedor', 'tiene_proyecto',
    'incoterm_familia', 'tarifa_hist_prov_concepto', 'tarifa_hist_concepto',
    'tarifa_hist_global', 'tarifa_historica', 'proveedor_frecuencia',
    'mes_sin', 'mes_cos', 'semana_sin', 'semana_cos', 'semanas_hasta_cierre',
    'es_cierre_fiscal', 'dias_transito_estimado', 'ruta_origen_grupo',
    'ratio_hist_concepto',
]
print(f"\n📋 Verificación de features ({len(FEATURES_ESPERADAS)} esperadas):")
faltantes = [f for f in FEATURES_ESPERADAS if f not in train.columns]
presentes = [f for f in FEATURES_ESPERADAS if f in train.columns]
for f in presentes:
    print(f"  ✓ {f}")
if faltantes:
    for f in faltantes:
        print(f"  ✗ FALTA: {f}")
    print(f"\n⚠️ {len(faltantes)} features faltantes — revisar notebook antes de entrenar")
else:
    print(f"\n✓ Todas las {len(FEATURES_ESPERADAS)} features presentes. Dataset listo para entrenar.")